# 04 · Results: tables, figures and significance tests (task T7)

Reads every saved run on Drive (full-data XGBoost from notebook 02, foundation models and matched XGBoost from notebook 03) and builds:

| File | What it is |
| --- | --- |
| `runs_all.csv` | one row per run (model × fraud rate × k × seed), every metric |
| `headline_table.csv` | mean ± std PR-AUC per model and fraud rate: the main comparison |
| `by_k_table.csv` | the same, for every context size k |
| `paired_tests.csv` | per seed, is the TFM's PR-AUC really different from full-data XGBoost? |
| `fig1_headline.png` | PR-AUC against fraud rate, both families |
| `fig2_by_k.png` | PR-AUC against context size, one panel per fraud rate |
| `fig3_context_frauds.png` | PR-AUC against how many frauds the context held |
| `policy_table_tabiclv2.csv`, `fig4_policies.png`, `context_variance.csv` | W1 context policies (after notebook 05) |

No model is run here, so a **CPU runtime** is enough. At the end everything is zipped and downloaded to your laptop.

In [ ]:
# 1. Drive, latest code, libraries.
from google.colab import drive
drive.mount("/content/drive")
import os, sys
REPO_URL = "https://github.com/Pakkabhabad18/tabular-fm-fraud-detection.git"
REPO_DIR = "/content/tabular-fm-fraud-detection"
if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
!pip install -q -r {REPO_DIR}/requirements.txt
sys.path.insert(0, f"{REPO_DIR}/src")

In [ ]:
# 2. Labels and split sizes (needed for the paired tests and the k rule).
from fraudicl.config import DRIVE_DIR
from fraudicl.data import load_ulb, features_and_target, load_splits, fingerprint
X, y = features_and_target(load_ulb())
splits = load_splits(f"{DRIVE_DIR}/splits/ulb_splits.npz")
expected = open(f"{REPO_DIR}/results/ulb_splits_fingerprint.txt").readline().strip()
assert fingerprint(splits) == expected, "splits on Drive differ from the ones on record"
train_rows = {s.level: len(s.train) for s in splits if s.seed == 0}
OUT = f"{DRIVE_DIR}/results"
print(train_rows)

In [ ]:
# 3. Every saved run in one table. Check how many seeds each model has so far:
#    the headline numbers are only final once every cell shows 5.
from fraudicl import report as R
runs = R.load_runs(OUT)
runs.groupby(["model", "level"], observed=True).seed.nunique().unstack()

In [ ]:
# 4. The headline comparison. Foundation models and matched XGBoost are shown at the
#    largest context size k that fits each fraud rate (fixed rule, never picked by
#    test score); full-data XGBoost uses every training row.
headline = R.headline_table(runs, train_rows)
headline.round(4)

In [ ]:
# 5. Paired bootstrap per seed: TFM (or matched XGBoost) minus full-data XGBoost,
#    on the same test rows. "worse/better" = the 95% interval of the difference
#    sits entirely below/above zero on that seed. Takes a few minutes.
tests = R.paired_tests(OUT, y, runs, train_rows, n_boot=1000)
tests.round(4)

In [ ]:
# 6. Figures and tables, written to /content/export.
import shutil
EXPORT = "/content/export"
os.makedirs(f"{EXPORT}/figures", exist_ok=True)
by_k = R.by_k_table(runs)
runs.to_csv(f"{EXPORT}/runs_all.csv", index=False)
headline.to_csv(f"{EXPORT}/headline_table.csv", index=False)
by_k.to_csv(f"{EXPORT}/by_k_table.csv", index=False)
tests.to_csv(f"{EXPORT}/paired_tests.csv", index=False)
R.plot_headline(headline, f"{EXPORT}/figures/fig1_headline.png")
R.plot_by_k(by_k, runs, f"{EXPORT}/figures/fig2_by_k.png")
R.plot_vs_context_frauds(runs, f"{EXPORT}/figures/fig3_context_frauds.png")
from IPython.display import Image, display
for f in ["fig1_headline", "fig2_by_k", "fig3_context_frauds"]:
    display(Image(f"{EXPORT}/figures/{f}.png"))

**Reading the figures.**

- **Figure 1**: where the orange/green lines (foundation models) sit against the blue one (XGBoost on all rows) at each fraud rate. The gap at 0.172% is what a uniform context costs. The dashed yellow line is XGBoost on the same k rows: if it sits on top of the TFM lines, the gap is about *how much data* each model sees, not the model family.
- **Figure 2**: whether a bigger context helps. If the TFM lines climb with k at 0.5% and 0.172%, the models are short of fraud examples, which is exactly what class-aware contexts (W1) fix.
- **Figure 3**: each point is one run at the natural rate. Look at the left end: how the models score with 1, 2 or 5 frauds in the context.

In [ ]:
# 6b. W1 context policies (only if notebook 05 has been run): ranking table and Figure 4.
if runs.policy.isin(["stratified", "balanced"]).any():
    policies = R.policy_table(runs, "TabICLv2")
    policies.to_csv(f"{EXPORT}/policy_table_tabiclv2.csv", index=False)
    R.plot_policies(policies, runs, f"{EXPORT}/figures/fig4_policies.png")
    cv_dir = f"{OUT}/raw/context_variance"
    if os.path.exists(cv_dir):
        import json, pandas as pd
        cv = pd.DataFrame([json.load(open(f"{cv_dir}/{f}")) for f in sorted(os.listdir(cv_dir))])
        cv.to_csv(f"{EXPORT}/context_variance.csv", index=False)
    display(Image(f"{EXPORT}/figures/fig4_policies.png"))
    display(policies.round(4))

In [ ]:
# 7. Zip and download. Move the zip into the repo folder on your laptop
#    (tabular-fm-fraud-detection/) and unzip it into results/.
shutil.make_archive("/content/results_export", "zip", EXPORT)
from google.colab import files
files.download("/content/results_export.zip")